# FlightAI Booking Assistant
A simple flight booking chatbot using llama3.2, Gradio, and SQLite.
The assistant collects the user's name, email, and destination, stores the booking, and lets them retrieve it.

In [ ]:
import json
import sqlite3
import gradio as gr
from openai import OpenAI

In [ ]:
MODEL = 'llama3.2'
ol = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
DB = 'bookings.db'

with sqlite3.connect(DB) as conn:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS bookings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            email TEXT NOT NULL,
            destination TEXT NOT NULL
        )
    ''')
    conn.commit()

print('DB ready.')

In [ ]:
ticket_prices = {"london": 799, "paris": 899, "tokyo": 1420, "berlin": 499, "sydney": 2999}

def get_ticket_price(destination):
    print(f'TOOL: get_ticket_price({destination})')
    price = ticket_prices.get(destination.lower())
    if price:
        return f"A ticket to {destination} costs ${price}."
    return f"Sorry, we don't have pricing for {destination} yet."

def store_booking(name, email, destination):
    print(f'TOOL: store_booking({name}, {email}, {destination})')
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO bookings (name, email, destination) VALUES (?, ?, ?)',
            (name, email, destination)
        )
        conn.commit()
    return f"Booking confirmed for {name} ({email}) to {destination}."

def get_booking(email):
    print(f'TOOL: get_booking({email})')
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            'SELECT name, destination FROM bookings WHERE email = ?', (email,)
        ).fetchall()
    if not rows:
        return f"No bookings found for {email}."
    results = ', '.join(f"{name} -> {dest}" for name, dest in rows)
    return f"Bookings for {email}: {results}"

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a ticket to a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination": {"type": "string", "description": "The destination city"}
                },
                "required": ["destination"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "store_booking",
            "description": "Store a confirmed flight booking. Only call this after you have the user's name, email, destination, AND the user has confirmed they want to book after seeing the price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name":        {"type": "string", "description": "The user's full name"},
                    "email":       {"type": "string", "description": "The user's email address"},
                    "destination": {"type": "string", "description": "The destination city"}
                },
                "required": ["name", "email", "destination"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_booking",
            "description": "Retrieve existing bookings for a user by their email address.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The user's email address"}
                },
                "required": ["email"],
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
SYSTEM = """
You are a flight booking assistant for FlightAI.

Follow these steps in order to book a flight:
1. Ask for the user's full name.
2. Ask for the user's email address.
3. Ask for their destination city.
4. Call get_ticket_price with the destination to fetch the price, then tell the user the price.
5. Ask if they want to confirm the booking.
6. If they confirm, call store_booking with their name, email, and destination.

Rules:
- Ask for ONE piece of information per message.
- Do NOT skip any step.
- Do NOT call store_booking before completing all steps 1-5.
- Do NOT make up or placeholder any values. Only use what the user actually typed.
- To retrieve an existing booking, ask for their email then call get_booking.
- Keep answers short.
"""

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if fn_name == 'get_ticket_price':
            if not args.get('destination'):
                result = "Missing destination. Ask the user for their destination city."
            else:
                result = get_ticket_price(args['destination'])

        elif fn_name == 'store_booking':
            missing = [f for f in ('name', 'email', 'destination') if not args.get(f)]
            if missing:
                result = f"Cannot book yet. Still missing: {', '.join(missing)}. Ask the user for these."
            else:
                result = store_booking(args['name'], args['email'], args['destination'])

        elif fn_name == 'get_booking':
            if not args.get('email'):
                result = "Missing email. Ask the user for their email address."
            else:
                result = get_booking(args['email'])
        else:
            result = 'Unknown tool.'

        responses.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return responses

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM}] + history + [{"role": "user", "content": message}]

    response = ol.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_msg)
        tool_content = " ".join(r["content"] for r in tool_responses)
        messages.append(assistant_msg)
        messages.extend(tool_responses)
        messages.append({"role": "user", "content": f"Tool result: {tool_content}. Please respond to the user now."})
        response = ol.chat.completions.create(model=MODEL, messages=messages, tool_choice="none")

    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages", title="FlightAI Booking Assistant").launch()